# 第1讲 课程代码案例：Python 数据处理 × Vibe Coding

> 课程：Python 进阶 | 教师：孙青 / 欧阳元新 | 平台：CloudStudio + CodeBuddy
>
> **AI 角色定位 = 编程搭档**：Vibe Coding 提示 → 迭代生成 → 代码纠错

## Part 1：Pandas Series 基础与 AI 协作

**受众**：有 Python 基础（列表、字典、循环），初次接触 Pandas 的学生

**前置条件**：已安装 pandas、numpy；了解 CloudStudio + CodeBuddy 基本操作

**学习目标**：
1. 掌握 Series 的三种创建方式及标签索引
2. 掌握 Series 统计方法全景（mean/std/describe/value_counts）
3. 理解 shift/rolling 在时间序列中的作用
4. 学会使用两段式 Prompt 与 AI 协作生成 Series 分析代码

### 1.1 Series 创建：三种方式

In [1]:
import pandas as pd
import numpy as np

# 方式一：列表（自动整数索引）
aqi_values = pd.Series([89, 45, 67, 112, 78, 55, 93])
print("方式一 - 列表创建:")
print(aqi_values)
print(f"dtype: {aqi_values.dtype}, 长度: {len(aqi_values)}")
print()

# 方式二：字典（键自动成为 index）—— 最推荐
aqi = pd.Series({
    '北京': 89, '上海': 45, '广州': 67,
    '成都': 112, '西安': 78, '深圳': 55, '杭州': 93
}, name='AQI指数')
print("方式二 - 字典创建:")
print(aqi)
print()

# 方式三：常量广播
baseline = pd.Series(75, index=['北京', '上海', '广州', '成都', '西安'])
print("方式三 - 常量广播:")
print(baseline)

方式一 - 列表创建:
0     89
1     45
2     67
3    112
4     78
5     55
6     93
dtype: int64
dtype: int64, 长度: 7

方式二 - 字典创建:
北京     89
上海     45
广州     67
成都    112
西安     78
深圳     55
杭州     93
Name: AQI指数, dtype: int64

方式三 - 常量广播:
北京    75
上海    75
广州    75
成都    75
西安    75
dtype: int64


### 1.2 Series 索引与筛选

In [2]:
# 标签索引
print(f"北京 AQI: {aqi['北京']}")
print(f"取多值:\n{aqi[['北京', '成都', '杭州']]}")
print()

# 条件筛选（布尔索引）
polluted = aqi[aqi > 75]
print(f"AQI > 75 的城市:\n{polluted}")
print()

# 排序
print(f"AQI 降序:\n{aqi.sort_values(ascending=False)}")

北京 AQI: 89
取多值:
北京     89
成都    112
杭州     93
Name: AQI指数, dtype: int64

AQI > 75 的城市:
北京     89
成都    112
西安     78
杭州     93
Name: AQI指数, dtype: int64

AQI 降序:
成都    112
杭州     93
北京     89
西安     78
广州     67
深圳     55
上海     45
Name: AQI指数, dtype: int64


### 1.3 Series 统计方法全景

In [3]:
print("=== AQI 统计分析 ===")
print(f"总和: {aqi.sum()}")
print(f"均值: {aqi.mean():.2f}")
print(f"标准差: {aqi.std():.2f}")
print(f"中位数: {aqi.median()}")
print(f"最大值: {aqi.max()} ({aqi.idxmax()})")
print(f"最小值: {aqi.min()} ({aqi.idxmin()})")
print()

# 一行搞定的描述性统计
print("=== describe() 摘要 ===")
print(aqi.describe())

=== AQI 统计分析 ===
总和: 539
均值: 77.00
标准差: 23.22
中位数: 78.0
最大值: 112 (成都)
最小值: 45 (上海)

=== describe() 摘要 ===
count      7.000000
mean      77.000000
std       23.216374
min       45.000000
25%       61.000000
50%       78.000000
75%       91.000000
max      112.000000
Name: AQI指数, dtype: float64


### 1.4 时间序列操作：shift 与 rolling

In [4]:
# 模拟月度数据（从航天贸易数据提取某国24个月进口额）
months = [f'2022-{i:02d}' for i in range(1, 13)] + [f'2023-{i:02d}' for i in range(1, 13)]
japan_import = pd.Series(
    [4523, 3891, 5102, 4678, 5234, 4012, 3567, 4890, 5678, 4321, 3998, 5432,
     4756, 4102, 5389, 4890, 5567, 4234, 3789, 5123, 5890, 4567, 4198, 5678],
    index=months, name='日本进口额(万元)'
)

# shift：环比变化（本月 - 上月）
mom_change = japan_import - japan_import.shift(1)
print("=== 环比变化（前5个月）===")
print(mom_change.head())
print()

# rolling：3个月移动平均（平滑短期波动）
rolling_avg = japan_import.rolling(3).mean()
print("=== 3个月移动平均（前5个月）===")
print(rolling_avg.head())

=== 环比变化（前5个月）===
2022-01       NaN
2022-02    -632.0
2022-03    1211.0
2022-04    -424.0
2022-05     556.0
Name: 日本进口额(万元), dtype: float64

=== 3个月移动平均（前5个月）===
2022-01            NaN
2022-02            NaN
2022-03    4505.333333
2022-04    4557.000000
2022-05    5004.666667
Name: 日本进口额(万元), dtype: float64


### 1.5 AI 协作练习：两段式 Prompt 生成 Series 分析

**课堂演示**：以下是一个两段式 Prompt 的示例及 AI 预期输出

```
【氛围段】我有7个城市的AQI数据，想快速了解哪些城市空气质量差、整体情况如何。

【约束段】请用 pandas Series 实现：
1. 用城市名做 index，数据为 {北京:89,上海:45,广州:67,成都:112,西安:78,深圳:55,杭州:93}
2. 打印 mean、std、idxmax，用中文注释说明含义
3. 用布尔索引筛选 AQI > 75 的城市
4. 变量名英文，注释中文
```

**审查要点**：AI 是否正确使用了 `idxmax()`？筛选方向对吗？

## Part 2：Pandas DataFrame 操作与分组聚合

**受众**：已掌握 Series 基础，首次接触 DataFrame 的学生

**前置条件**：Part 1 完成

**学习目标**：
1. 掌握 DataFrame 创建、列行操作与条件筛选
2. 掌握 groupby 分组聚合与 melt 宽转长
3. 学会配置 Rules 文件规范 AI 输出
4. 学会使用 Diff 三步审查清单

### 2.1 DataFrame 创建与基本操作

In [5]:
# 用字典创建 DataFrame
products = pd.DataFrame({
    '产品': ['耳机', '键盘', '鼠标', '显示器', '摄像头', '麦克风'],
    '一季度': [1200, 980, 1560, 2300, 890, 670],
    '二季度': [1450, 1120, 1820, 2100, 1040, 750],
    '三季度': [1680, 890, 2010, 2450, 960, 820]
})
print(f"形状: {products.shape}")
print(f"列: {list(products.columns)}")
print()
print(products)

形状: (6, 4)
列: ['产品', '一季度', '二季度', '三季度']

    产品   一季度   二季度   三季度
0   耳机  1200  1450  1680
1   键盘   980  1120   890
2   鼠标  1560  1820  2010
3  显示器  2300  2100  2450
4  摄像头   890  1040   960
5  麦克风   670   750   820


### 2.2 列操作与向量化计算

In [6]:
# 新增列（向量化，禁止 for 循环）
products['总销量'] = products['一季度'] + products['二季度'] + products['三季度']

# 条件映射
conditions = [products['总销量'] >= 6000, products['总销量'] >= 4000]
products['等级'] = np.select(conditions, ['A', 'B'], default='C')

# Top N
print("=== 总销量 Top 3 ===")
print(products.nlargest(3, '总销量')[['产品', '总销量', '等级']])

=== 总销量 Top 3 ===
    产品   总销量 等级
3  显示器  6850  A
2   鼠标  5390  B
0   耳机  4330  B


### 2.3 读取真实数据：航天进出口额

In [8]:
# 读取航天进出口贸易数据
df = pd.read_excel('./实验一/航天进出口额.xlsx', sheet_name='Sheet0')
print(f"数据形状: {df.shape}")
print(f"列数: {len(df.columns)}")
print()
print("前3行预览:")
print(df.head(3))

数据形状: (76, 29)
列数: 29

前3行预览:
                                    指标 进出口  地区 频度  单位  2022-01  2022-02  \
0    进口额(人民币)_(HS88章)航空器、航天器及其零件_印度_当期  进口  全国  月  万元   288.13   601.83   
1  进口额(人民币)_(HS88章)航空器、航天器及其零件_中国香港_当期  进口  全国  月  万元     5.54     1.52   
2    进口额(人民币)_(HS88章)航空器、航天器及其零件_日本_当期  进口  全国  月  万元  2653.87  1473.53   

   2022-03  2022-04  2022-05  ...  2023-03  2023-04  2023-05  2023-06  \
0   277.40    937.0   460.15  ...   558.96   603.73   309.91   240.31   
1      NaN      NaN     0.92  ...    14.58     2.05     4.38     0.14   
2  1739.32    690.0  2964.17  ...  1968.32  4095.97  1971.03  1433.03   

   2023-07  2023-08  2023-09  2023-10  2023-11  2023-12  
0   164.48   398.07   540.10   608.22   827.64   795.53  
1      NaN     1.41     2.33     6.13    15.03     4.99  
2   183.06  1970.47  2156.61  1691.27  1367.80  3274.18  

[3 rows x 29 columns]


### 2.4 groupby 分组聚合

In [9]:
# 解析指标列：提取方向和国家
first_col = df.columns[0]
df['方向'] = df[first_col].apply(lambda x: '进口' if '进口' in str(x) else '出口')
df['国家'] = df[first_col].apply(lambda x: str(x).split('_')[-2] if '_' in str(x) else '未知')

# 月度数据列
month_cols = [c for c in df.columns if c not in [first_col, '方向', '国家']]

# 按方向分组统计
df['年度合计'] = df[month_cols].sum(axis=1)
direction_stats = df.groupby('方向')['年度合计'].agg(['sum', 'mean', 'count'])
print("=== 按方向分组统计 ===")
print(direction_stats)
print()

# 进口 Top5 来源国
df_import = df[df['方向'] == '进口'].copy()
top5 = df_import.nlargest(5, '年度合计')[['国家', '年度合计']]
print("=== 进口额 Top5 国家 ===")
print(top5.to_string(index=False))

TypeError: can only concatenate str (not "float") to str

### 2.5 melt 宽转长：为时间序列分析做准备

In [ ]:
# 宽表 → 长表
df_long = df_import[['国家'] + month_cols[:6]].head(5).melt(
    id_vars=['国家'],
    value_vars=month_cols[:6],
    var_name='月份',
    value_name='贸易额'
)
print(f"宽表形状: (5, 7) → 长表形状: {df_long.shape}")
print()
print("长表前10行:")
print(df_long.head(10))

### 2.6 AI 协作练习：Rules + Diff 审查

**Rules 文件示例** (`.codebuddy/rules/python-style.md`)：

```markdown
# Python 数据分析编码规范
- 变量名：英文，snake_case
- 注释：全部中文
- 禁止 for 循环逐行处理列（必须用向量化操作）
- DataFrame 修改必须用 .copy() 避免 SettingWithCopyWarning
- 含中文的 CSV 必须加 encoding='utf-8-sig'
```

**Diff 审查三步清单**：
1. **结构**：导入是否齐全？命名是否一致？
2. **逻辑**：数据流是否正确？筛选条件是否遗漏？
3. **边界**：空值是否处理？类型是否匹配？

## Part 3：数据文件操作与上下文工程

**受众**：需要处理多格式数据文件的学生

**前置条件**：Part 1-2 完成

**学习目标**：
1. 掌握 CSV/Excel/JSON 三种格式的读写
2. 理解缺失值基本策略（dropna vs fillna）
3. 掌握上下文四层级与 @文件引用

### 3.1 多格式文件读写

In [ ]:
# CSV 读写（注意中文编码）
products.to_csv('output/products.csv', index=False, encoding='utf-8-sig')
df_csv = pd.read_csv('output/products.csv', encoding='utf-8-sig')
print(f"CSV 读取成功: {df_csv.shape}")
print()

# Excel 多 sheet 探索
xls = pd.ExcelFile('航天进出口额.xlsx')
print(f"Excel 包含的 Sheet: {xls.sheet_names}")
print()

# JSON 读写
products.to_json('output/products.json', orient='records', force_ascii=False, indent=2)
df_json = pd.read_json('output/products.json')
print(f"JSON 读取成功: {df_json.shape}")

### 3.2 缺失值分析与处理策略

In [ ]:
# 航天数据缺失值分析
missing = df[month_cols].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'缺失数': missing, '缺失率%': missing_pct})
missing_report = missing_report[missing_report['缺失数'] > 0]
print(f"=== 含缺失值的列（共{len(missing_report)}列）===")
print(missing_report)
print()

# 策略对比
print("=== 缺失值策略对比 ===")
print(f"dropna 后行数: {df.dropna(subset=month_cols).shape[0]} (原始{len(df)}行)")
print(f"fillna(中位数) 后缺失: {df[month_cols].fillna(df[month_cols].median()).isnull().sum().sum()}")

### 3.3 上下文工程：让 AI 精准理解你的数据

**上下文四层级**：

| 层级 | 来源 | 控制方式 |
|------|------|---------|
| 系统层 | Rules 文件 | 编辑 `.codebuddy/rules/` |
| 会话层 | 对话历史 | 开新对话清空 |
| 指令层 | @引用 | 主动选择文件 |
| 隐式层 | 当前打开文件 | 切换文件 |

**@文件引用示例**：

```
@航天进出口额.xlsx 读取这个文件，展示前5行和列名，
告诉我指标列的命名规则是什么
```

**最小上下文原则**：信息不足→AI瞎猜 | 信息过多→AI混乱 | 信息精准→代码直接可用

## Part 4：数据清洗与描述性统计

**受众**：需要处理真实"脏数据"的学生

**前置条件**：Part 1-3 完成

**学习目标**：
1. 掌握重复值检测与异常值识别（IQR / Z-score）
2. 掌握数据类型转换
3. 掌握描述性统计与相关性分析
4. 学会使用 Inline Edit + Diff 审查优化 AI 生成的清洗代码

### 4.1 重复值检测

In [ ]:
# 检查重复行
dup_count = df.duplicated().sum()
print(f"重复行数: {dup_count}")

# 如果有重复，展示并删除
if dup_count > 0:
    print("重复行示例:")
    print(df[df.duplicated(keep=False)].head())
    df_clean = df.drop_duplicates()
    print(f"去重后: {len(df)} → {len(df_clean)} 行")
else:
    df_clean = df.copy()
    print("无重复行，数据完整性良好")

### 4.2 异常值识别：IQR 法与 Z-score

In [ ]:
from scipy import stats

# 选取一个数值列做演示
sample_col = month_cols[0]
values = df_clean[sample_col].dropna()
print(f"分析列: {sample_col}, 非空值数: {len(values)}")
print()

# IQR 法（适合偏态数据）
Q1 = values.quantile(0.25)
Q3 = values.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers_iqr = values[(values < lower) | (values > upper)]
print(f"=== IQR 法 ===")
print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"正常范围: [{lower:.2f}, {upper:.2f}]")
print(f"异常值数量: {len(outliers_iqr)}")
print()

# Z-score 法（适合近似正态）
z_scores = np.abs(stats.zscore(values))
outliers_z = values[z_scores > 3]
print(f"=== Z-score 法（阈值=3）===")
print(f"异常值数量: {len(outliers_z)}")
if len(outliers_z) > 0:
    print(f"异常值: {outliers_z.values[:5]}")

### 4.3 数据类型转换

In [ ]:
# 查看当前类型
print("=== 数据类型概览 ===")
print(df_clean.dtypes.value_counts())
print()

# 分类类型优化内存
df_clean['方向'] = df_clean['方向'].astype('category')
print(f"'方向'列转为 category: {df_clean['方向'].dtype}")

# 数值类型确认
numeric_cols = df_clean.select_dtypes(include='number').columns
print(f"数值列数: {len(numeric_cols)}")

### 4.4 描述性统计全景

In [ ]:
# 对进口数据做描述性统计
import_data = df_clean[df_clean['方向'] == '进口'][month_cols]
stats_summary = import_data.describe()
print("=== 进口数据月度统计摘要（前6列）===")
print(stats_summary[month_cols[:6]])
print()

# 偏度和峰度
sample = import_data[month_cols[0]].dropna()
print(f"=== {month_cols[0]} 分布特征 ===")
print(f"偏度(skew): {sample.skew():.4f} {'→ 右偏' if sample.skew() > 0 else '→ 左偏'}")
print(f"峰度(kurtosis): {sample.kurtosis():.4f} {'→ 尖峰' if sample.kurtosis() > 0 else '→ 平坦'}")

### 4.5 相关性分析

In [ ]:
# 计算月度贸易额之间的相关系数
corr_matrix = import_data[month_cols[:12]].corr()

# 找出高相关的月份对（>0.8）
high_corr = []
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            high_corr.append((corr_matrix.index[i], corr_matrix.columns[j], r))

print("=== 高相关月份对（|r| > 0.8）===")
if high_corr:
    for m1, m2, r in sorted(high_corr, key=lambda x: -abs(x[2]))[:5]:
        print(f"  {m1} ↔ {m2}: r = {r:.4f}")
else:
    print("  未发现 |r| > 0.8 的月份对")

### 4.6 AI 协作练习：Inline Edit + Diff 审查

**场景**：AI 生成了一段清洗代码，你需要审查并修正

**Inline Edit 使用方法**：`Cmd+K`（Mac）/ `Ctrl+K`（Win）在光标位置唤起编辑

**典型修正场景**：
- AI 用了 `dropna()` 但你想改为 `fillna(median())`
- AI 的异常值阈值设成 2（太松），你改为 3
- AI 忘了 `encoding='utf-8-sig'`

**Diff 三步清单实战**：

```
Round 1 Prompt：
@航天进出口额.xlsx 对这个数据做完整清洗：检查重复→解析指标列→异常值检测→缺失值填充

→ AI 输出代码 → 按三步审查：
1. 结构：导入 scipy 了吗？变量名一致吗？
2. 逻辑：split 分隔符对吗？IQR 公式正确吗？
3. 边界：全 NaN 列怎么办？数值为 0 是异常还是正常？
```

## Part 5：小结

**受众**：全体学生

**前置条件**：Part 1-4 完成

**学习目标**：
1. 回顾本讲核心知识点
2. 掌握 Vibe Coding 工具用法

### 知识点速查表

In [ ]:
# 本讲核心操作速查（可直接复制使用）
quick_ref = {
    '创建 Series': "pd.Series({'北京': 89, '上海': 45})",
    '创建 DataFrame': "pd.DataFrame({'col': [1,2,3]})",
    '读 Excel': "pd.read_excel('f.xlsx', sheet_name='Sheet0')",
    '条件筛选': "df[df['方向'] == '进口']",
    'Top N': "df.nlargest(10, '年度合计')",
    '分组聚合': "df.groupby('方向').agg(['sum','mean'])",
    '宽转长': "df.melt(id_vars=['国家'], value_vars=cols)",
    '缺失值填充': "df.fillna(df.median())",
    '重复值': "df.drop_duplicates()",
    '异常值 IQR': "Q1 - 1.5*IQR ~ Q3 + 1.5*IQR",
    '相关性': "df.corr()",
    '描述统计': "df.describe() / .skew() / .kurtosis()",
}

print("=" * 50)
print("第1讲 知识点速查表")
print("=" * 50)
for op, code_str in quick_ref.items():
    print(f"  {op:12s} → {code_str}")

### Vibe Coding 工具速查表

In [ ]:
tools_ref = {
    '两段式 Prompt': '先描述氛围（模糊意图），再逐步加约束',
    'Rules 文件': '锁定 AI 输出风格（命名/注释/禁止项）',
    '@文件引用': '精准注入数据结构，减少重复描述',
    'Inline Edit': 'Cmd+K 行内修改，不开新对话',
    'Diff 审查': '三步清单：结构→逻辑→边界',
    '上下文四层级': '系统/会话/指令/隐式 — 精准 > 全面',
}

print("=" * 50)
print("Vibe Coding 工具速查表")
print("=" * 50)
for tool, desc in tools_ref.items():
    print(f"  {tool:12s} → {desc}")
print()
print("AI 角色：编程搭档 — 提示生成 / 迭代修改 / 代码纠错")